# BÁO CÁO PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA - EXPLORATORY DATA ANALYSIS)
## Đề tài: Tìm hiểu thuật toán KNN và ứng dụng hệ gợi ý
**Người thực hiện:** Linh (Phụ trách Module Tiền xử lý & EDA)
**Môn học:** Mạng máy tính (MMT)

---
### Mục tiêu nghiên cứu của Notebook:
1. Khảo sát cấu trúc tập dữ liệu tương tác người dùng - vật phẩm (ratings).
2. Phân tích phân phối tần suất điểm đánh giá (Rating Distribution).
3. Phân tích phân phối đuôi dài (Long-tail distribution) của User và Item.
4. Đánh giá tác động của kỹ thuật lọc lặp K-Core (ngưỡng tối thiểu 5 tương tác).
5. Tính toán chính xác độ thưa (Sparsity) và mật độ (Density) của ma trận User-Item.
6. Thử nghiệm và so sánh các phương pháp chuẩn hóa dữ liệu: Min-Max Scaling và Z-Score Standardization.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập thư mục gốc dự án
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.preprocess import (
    load_raw_ratings,
    filter_k_core,
    build_user_item_matrix,
    calculate_sparsity,
    min_max_scale,
    z_score_scale,
    mean_center,
    summary_statistics
)

# Cấu hình hiển thị đồ thị
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (10, 5)
print("Đã nạp thành công các thư viện và module tiền xử lý của Linh!")

## 1. Nạp và kiểm tra dữ liệu tương tác thô
Ta kiểm tra file `data/raw/ml-latest-small/ratings.csv`. Nếu chưa có dữ liệu thật từ GroupLens, ta tự động sinh tập dữ liệu mô phỏng 600 user và 1,500 phim tuân theo phân phối Zipf (đúng bản chất thực tế) để tiến hành phân tích.

In [ ]:
raw_path = os.path.join(project_root, 'data', 'raw', 'ml-latest-small', 'ratings.csv')

if os.path.exists(raw_path):
    print(f"Nạp dữ liệu thật từ: {raw_path}")
    df_ratings = load_raw_ratings(raw_path)
else:
    print("Chưa có file ratings.csv trong data/raw/ml-latest-small/. Tự động sinh dữ liệu thực nghiệm mô phỏng Zipf...")
    np.random.seed(42)
    n_sim_users = 600
    n_sim_items = 1500
    n_sim_ratings = 80000
    
    # Mô phỏng phân phối đuôi dài Zipf
    user_pop = np.random.zipf(a=1.5, size=n_sim_ratings) % n_sim_users + 1
    item_pop = np.random.zipf(a=1.4, size=n_sim_ratings) % n_sim_items + 1
    # Rating thực tế tập trung nhiều vào mức 3.0, 4.0, 5.0
    ratings_pool = np.random.choice([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], 
                                    size=n_sim_ratings, 
                                    p=[0.02, 0.04, 0.03, 0.07, 0.09, 0.18, 0.15, 0.25, 0.08, 0.09])
    
    df_ratings = pd.DataFrame({
        'userId': user_pop,
        'movieId': item_pop,
        'rating': ratings_pool
    }).drop_duplicates(subset=['userId', 'movieId'])

print(f"Tổng số dòng dữ liệu: {len(df_ratings):,}")
print(f"Số lượng Users: {df_ratings['userId'].nunique():,}")
print(f"Số lượng Items: {df_ratings['movieId'].nunique():,}")
df_ratings.head(10)

## 2. Phân tích phân phối điểm đánh giá (Rating Distribution)
Khảo sát xu hướng chấm điểm của người dùng để nắm bắt độ lệch (skewness) của dữ liệu.

In [ ]:
stats = summary_statistics(df_ratings, rating_col='rating')
print("=== BẢNG THỐNG KÊ MÔ TẢ ĐIỂM ĐÁNH GIÁ ===")
for k, v in stats.items():
    print(f"{k:15}: {v:.4f}" if isinstance(v, float) else f"{k:15}: {v:,}")

# Trực quan hóa phân phối rating
plt.figure(figsize=(9, 4.5))
counts = df_ratings['rating'].value_counts().sort_index()
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(counts)))
bars = plt.bar(counts.index.astype(str), counts.values, color=colors, edgecolor='black', alpha=0.85)

plt.title('Phân phối tần suất điểm đánh giá (Rating Distribution)', fontsize=13, fontweight='bold')
plt.xlabel('Điểm đánh giá (Rating)', fontsize=11)
plt.ylabel('Số lượng tương tác', fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Hiển thị nhãn số lượng trên đầu cột
for bar in bars:
    height = bar.get_height()
    plt.annotate(f'{height:,}',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3), textcoords="offset points",
                 ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Hiện tượng đuôi dài (Long-Tail Distribution)
Trong hầu hết các hệ thống thực tế:
- Một số ít user cực kỳ tích cực (chấm hàng trăm phim), trong khi đại đa số chỉ chấm vài lần.
- Một số ít bộ phim bom tấn nhận được hàng nghìn lượt rating (Short Head), phần lớn các phim còn lại có rất ít lượt tương tác (Long Tail).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Số tương tác theo từng user
user_counts = df_ratings['userId'].value_counts().sort_values(ascending=False).values
axes[0].plot(range(len(user_counts)), user_counts, color='royalblue', lw=2)
axes[0].fill_between(range(len(user_counts)), user_counts, color='royalblue', alpha=0.2)
axes[0].set_title('Phân phối hoạt động của User (User Activity)', fontweight='bold')
axes[0].set_xlabel('Thứ hạng User (sắp xếp giảm dần)')
axes[0].set_ylabel('Số lượng lượt đánh giá')
axes[0].axhline(y=5, color='red', linestyle='--', label='Ngưỡng lọc K-core (min=5)')
axes[0].legend()

# Biểu đồ 2: Số tương tác theo từng item (phim)
item_counts = df_ratings['movieId'].value_counts().sort_values(ascending=False).values
axes[1].plot(range(len(item_counts)), item_counts, color='darkorange', lw=2)
axes[1].fill_between(range(len(item_counts)), item_counts, color='darkorange', alpha=0.2)
axes[1].set_title('Độ phổ biến của Vật phẩm (Item Popularity - Long Tail)', fontweight='bold')
axes[1].set_xlabel('Thứ hạng Item (sắp xếp giảm dần)')
axes[1].set_ylabel('Số lượng lượt đánh giá')
axes[1].axhline(y=5, color='red', linestyle='--', label='Ngưỡng lọc K-core (min=5)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Tác động của lọc lặp K-Core (User & Item >= 5 tương tác)
Lọc lặp k-core loại bỏ các đỉnh có bậc nhỏ hơn 5 trong đồ thị lưỡng phân User-Item, giúp:
- Hạn chế ma trận quá thưa và giảm nhiễu khi tính độ tương đồng Pearson/Cosine.
- Tiết kiệm bộ nhớ RAM khi tính ma trận tương quan.

In [ ]:
df_filtered = filter_k_core(df_ratings, min_user=5, min_item=5)

print("=== SO SÁNH TRƯỚC VÀ SAU LỌC K-CORE ===")
print(f"Trước lọc: {len(df_ratings):,} ratings | {df_ratings['userId'].nunique():,} users | {df_ratings['movieId'].nunique():,} items")
print(f"Sau lọc  : {len(df_filtered):,} ratings | {df_filtered['userId'].nunique():,} users | {df_filtered['movieId'].nunique():,} items")
print(f"Tỷ lệ bản ghi giữ lại: {len(df_filtered)/len(df_ratings)*100:.2f}%")

## 5. Dựng ma trận User-Item & Tính toán độ thưa (Sparsity)
Ta chuyển bảng sang ma trận 2 chiều $R_{m \times n}$ và tính toán độ thưa theo công thức:
$$\text{Sparsity} = \left(1 - \frac{|\text{Ratings}|}{|\text{Users}| \times |\text{Items}|}\right) \times 100\%$$

In [ ]:
R, user_ids, item_ids = build_user_item_matrix(df_filtered, min_user=5, min_item=5)
sp_info = calculate_sparsity(R)

print("=== KẾT QUẢ TÍNH ĐỘ THƯA MA TRẬN USER-ITEM ===")
for k, v in sp_info.items():
    print(f"{k:18}: {v}")

# Trực quan hóa một lát cắt nhỏ của ma trận (50 users x 50 items)
sample_R = R[:50, :50]
plt.figure(figsize=(8, 7))
sns.heatmap(sample_R != 0, cmap=['#f0f2f6', '#1f77b4'], cbar=False)
plt.title('Minh họa độ thưa trên lát cắt 50x50 của Ma trận User-Item\n(Màu xanh: Đã đánh giá | Màu xám: Ô trống / Unrated)', fontsize=11, fontweight='bold')
plt.xlabel('50 Items mẫu')
plt.ylabel('50 Users mẫu')
plt.tight_layout()
plt.show()

## 6. Thử nghiệm các phép chuẩn hóa: Min-Max và Z-Score
So sánh giá trị trước và sau khi chuẩn hóa trên tập các ô đã đánh giá.

In [ ]:
R_minmax = min_max_scale(R, feature_range=(0.0, 1.0), mask_zeros=True)
R_zscore = z_score_scale(R, axis=0, mask_zeros=True)
R_centered = mean_center(R, axis=0, mask_zeros=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Gốc
axes[0].hist(R[R > 0], bins=9, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Điểm đánh giá gốc (0.5 - 5.0)', fontweight='bold')
axes[0].set_xlabel('Rating')

# 2. Min-Max
axes[1].hist(R_minmax[R_minmax > 0], bins=9, color='teal', edgecolor='black', alpha=0.8)
axes[1].set_title('Chuẩn hóa Min-Max [0.0 - 1.0]', fontweight='bold')
axes[1].set_xlabel('Min-Max Scaled')

# 3. Z-Score
axes[2].hist(R_zscore[R_zscore != 0], bins=15, color='coral', edgecolor='black', alpha=0.8)
axes[2].set_title('Chuẩn hóa Z-Score theo User (Mean=0, Std=1)', fontweight='bold')
axes[2].set_xlabel('Z-Score')

plt.tight_layout()
plt.show()

print("Hoàn thành toàn bộ quy trình EDA của Linh!")